In [ ]:
# For API calls and response parsin
import pandas as pd
import requests
from pprint import pprint

# For connecting to Supabase
from sqlalchemy import create_engine
from sqlalchemy.dialects.postgresql import JSONB
from dotenv import load_dotenv
import os

# For parsing responses
from datetime import datetime, timezone
import json

In [ ]:
url = "https://clinicaltrials.gov/api/v2/studies"
params={"query.cond": "non-small cell lung cancer",
        "pageSize": 100,
        "filter.advanced": "AREA[StartDate]2022"}
response = requests.get(url, params)
study = response.json()

# Raw Layer

### Preparing the Table

In [ ]:
RAW_COLUMNS = ["nct_id", "raw_json", "fetched_at", "params", "response_url"]

def raw_builder(response: requests.Response, params: dict) -> pd.DataFrame:
    """Build raw-layer rows (one per study) from a ClinicalTrials.gov API response."""
    response.raise_for_status()
    studies = response.json()["studies"]

    fetched_at = datetime.now(timezone.utc)
    params = dict(params)  # snapshot, so later mutation doesn't leak into stored rows

    return pd.DataFrame(
        [
            {
                "nct_id": s["protocolSection"]["identificationModule"]["nctId"],
                "raw_json": s,
                "fetched_at": fetched_at,
                "params": params,
                "response_url": response.url,
            }
            for s in studies
        ],
        columns=RAW_COLUMNS,
    )

### Writing it To Supabase

In [ ]:
client = create_engine(os.getenv("SUPABASE_PROJECT_URL"))

raw_table = raw_builder(response, params)

# Study Tables

## Study Master

Columns:
  - [x] nct_id (primary key)
  - [x] study_title
  - [x] study_long_title
  - [x] study_acronym
  - [] sponsor_id (foreign key)
  - [] compound_id (foreign key)
  - [] company_id (foreign key)
  - [x] lead_sponsor
  - [] collaborator_id
  - [x] collaborator_names
  - [x] study_type
  - [x] conditions

Function:
  - parse_study_master --> takes the JSON response as input, flattens and extracts the variables in to a table

In [ ]:
# Flattener func
def extract_study_master(study: dict) -> dict:
    '''Flattens the study dictionary of the JSON API response into a single dictionary to turn into a DataFrame'''
    id_mod = study['protocolSection'].get("identificationModule", {})
    sponsor_mod = study['protocolSection'].get("sponsorCollaboratorsModule", {})
    cond_mod = study['protocolSection'].get("conditionsModule", {})
    outcome_mod = study['protocolSection'].get("outcomesModule", {})
    status_mod = study['protocolSection'].get("statusModule", {})
    design_mod = study['protocolSection'].get("designModule", {})

    extracted_dict = {
        "nct" : id_mod.get("nctId"),
        "study_title": id_mod.get("briefTitle"),
        "study_long_title": id_mod.get("officialTitle"),
        "study_acronym": id_mod.get("acronym"),
        "lead_sponsor": sponsor_mod.get("leadSponsor", {}).get("name"),
        "collaborators:": parse_collaborators(sponsor_mod),
        "endpoints": outcome_mod.get("outcomesModule"),
        "status": status_mod.get("overallStatus"),
        "study_type": design_mod.get('studyType'),
        "conditions": cond_mod.get("conditions")
    }
    
    return extracted_dict

# Parser for extracting collaborators
def parse_collaborators(x):
    '''Extracts the collaborators from the collaborators portion of the protocolSection'''
    try:
        return ': '.join(a.get('name') for a in x.get("collaborators"))
    except:
        return None

In [ ]:
test_results = pd.DataFrame.from_dict([extract_study_master(s) for s in study['studies']])

# Sponsor Tables

## Sponsor Master